# EARLI: VRPTW Training & Comparison
# PPO vs Tree-Based Pipeline

## Setup

### Install

Run the following steps **only if EARLI is not yet installed**.

In [ ]:
# Verify Python>=3.10
import sys
print(sys.version)

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Install build tools (only if not already installed)
# !apt-get update -y && apt-get install -y git build-essential ninja-build

In [ ]:
!pip install --upgrade --extra-index-url https://pypi.nvidia.com -c constraints.txt .

### Import

In [ ]:
import os
import pickle as pkl
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
import yaml
import torch

from earli.train import train
from earli import main as earli_main
from earli.cuopt_injection_vrptw import main as cuopt_main
from earli.utils import analysis_utils as utils
from earli.utils.nv import verify_consistent_config
from earli.vrptw import VRPTW

## Prepare Datasets

**Training data**: Homberger 200-customer benchmark (small scale).

**Test data**: Homberger 1000-customer benchmark — used **exclusively for testing**, never for training or validation.

In [ ]:
import os
os.makedirs('datasets', exist_ok=True)

# Parse 200-customer Homberger benchmark -> training dataset
!python -m earli.benchmark_parser vrptw homberger/homberger_200_customer_instances \
        datasets/vrptw_200.pkl

# Parse 1000-customer Homberger benchmark -> test-only dataset
# IMPORTANT: these instances must NOT be used for training or validation.
!python -m earli.benchmark_parser vrptw homberger/homberger_1000_customer_instances \
        datasets/vrptw_1000.pkl

In [ ]:
!ls -lh datasets/vrptw_200.pkl datasets/vrptw_1000.pkl

## Train

Two models are trained using the same n=200 training data:
1. **PPO** - Stable-Baselines3 on-policy rollout training.
2. **tree_based** - Tree-search guided PPO (higher data quality).

### 1. Train with PPO (Stable-Baselines3)

In [ ]:
# Config overview: config_vrptw_ppo_train.yaml
# method=ppo, data=vrptw_200.pkl, save=model_vrptw_ppo.m
with open('config_vrptw_ppo_train.yaml') as f:
    print(f.read())

In [ ]:
%%time
# Train PPO model on n=200 Homberger instances
train(config_path='config_vrptw_ppo_train.yaml')

### 2. Train with Tree-Based Pipeline

In [ ]:
# Config overview: config_vrptw_train.yaml
# method=tree_based, data=vrptw_200.pkl, save=model_vrptw.m
with open('config_vrptw_train.yaml') as f:
    print(f.read())

In [ ]:
%%time
# Train tree_based model on n=200 Homberger instances
train(config_path='config_vrptw_train.yaml')

## Test on n=1000 Instances

The n=1000 Homberger instances are used **exclusively** for testing.

### Test PPO model on n=1000

In [ ]:
import yaml

# Point the test config to the PPO model
with open('config_vrptw_test_1000.yaml') as f:
    test_cfg = yaml.safe_load(f)

ppo_model_path = 'earli/pretrained_models/model_vrptw_ppo.m'
test_cfg['train']['pretrained_fname'] = ppo_model_path
test_cfg['logger']['logging_level'] = 20  # verbose

tmp_cfg_ppo = 'config_vrptw_test_1000_ppo_tmp.yaml'
with open(tmp_cfg_ppo, 'w') as f:
    yaml.dump(test_cfg, f)

print('Testing PPO model on n=1000 instances ...')

In [ ]:
%%time
earli_main.main(config_path=tmp_cfg_ppo)

import shutil
shutil.copy('outputs/test_logs.pkl', 'outputs/test_logs_vrptw_1000_ppo.pkl')
print('PPO test log saved to outputs/test_logs_vrptw_1000_ppo.pkl')

### Test tree_based model on n=1000

In [ ]:
tree_model_path = 'earli/pretrained_models/model_vrptw.m'

with open('config_vrptw_test_1000.yaml') as f:
    test_cfg_tree = yaml.safe_load(f)

test_cfg_tree['train']['pretrained_fname'] = tree_model_path
test_cfg_tree['logger']['logging_level'] = 20

tmp_cfg_tree = 'config_vrptw_test_1000_tree_tmp.yaml'
with open(tmp_cfg_tree, 'w') as f:
    yaml.dump(test_cfg_tree, f)

print('Testing tree_based model on n=1000 instances ...')

In [ ]:
%%time
earli_main.main(config_path=tmp_cfg_tree)

import shutil
shutil.copy('outputs/test_logs.pkl', 'outputs/test_logs_vrptw_1000_tree.pkl')
print('Tree-based test log saved to outputs/test_logs_vrptw_1000_tree.pkl')

## Analyze Test Results

### Load Results

In [ ]:
with open('outputs/test_logs_vrptw_1000_ppo.pkl', 'rb') as f:
    ppo_log = pkl.load(f)

with open('outputs/test_logs_vrptw_1000_tree.pkl', 'rb') as f:
    tree_log = pkl.load(f)

with open('datasets/vrptw_1000.pkl', 'rb') as f:
    problems = pkl.load(f)

### Calculate Average Costs

In [ ]:
def compute_df(log, label, problems):
    best_sols = log['optimal_route']
    n = len(best_sols)
    vehicles = [int(np.sum([node == 0 for node in sol])) - 1 for sol in best_sols]
    dist_mats = problems['distance_matrix']
    costs = [utils.solution_cost(sol, dm).item() for sol, dm in zip(best_sols, dist_mats)]
    return pd.DataFrame(dict(method=label, problem_id=np.arange(n), vehicles=vehicles, cost=costs))

df_ppo  = compute_df(ppo_log,  'PPO',        problems)
df_tree = compute_df(tree_log, 'Tree-Based', problems)
df = pd.concat([df_ppo, df_tree], ignore_index=True)
df

In [ ]:
print('=== PPO model (n=1000) ===')
print(f'  Mean vehicles: {df_ppo.vehicles.mean():.2f}')
print(f'  Mean cost:     {df_ppo.cost.mean():.4f}')
print()
print('=== Tree-Based model (n=1000) ===')
print(f'  Mean vehicles: {df_tree.vehicles.mean():.2f}')
print(f'  Mean cost:     {df_tree.cost.mean():.4f}')

### Compare PPO vs Tree-Based

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

sns.barplot(data=df, x='method', y='vehicles', ax=axes[0])
axes[0].set_title('Mean # Vehicles (VRPTW n=1000)')

sns.barplot(data=df, x='method', y='cost', ax=axes[1])
axes[1].set_title('Mean Cost (VRPTW n=1000)')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/vrptw_1000_comparison.png', dpi=120)
plt.show()

cost_ratio = df_ppo.cost.mean() / df_tree.cost.mean()
print(f'Average PPO cost gap from Tree-Based: {100*(cost_ratio-1):.2f}%')

## cuOpt Injection

Run cuOpt on the n=1000 test instances, seeded with the RL solutions as an initial population. This section extends the comparison from *ExampleTrain.ipynb* to show results for both RL methods.

### cuOpt with PPO initialization

In [ ]:
import yaml

with open('config_vrptw_cuopt_1000.yaml') as f:
    cuopt_cfg_ppo = yaml.safe_load(f)

cuopt_cfg_ppo['paths']['solutions']   = 'outputs/test_logs_vrptw_1000_ppo.pkl'
cuopt_cfg_ppo['names']['methods']      = ['PPO+cuOpt']
cuopt_cfg_ppo['names']['main_method']  = 'PPO+cuOpt'
cuopt_cfg_ppo['paths']['out_summary']  = 'test_summary_vrptw_1000_ppo'

tmp_cuopt_ppo = 'config_vrptw_cuopt_1000_ppo_tmp.yaml'
with open(tmp_cuopt_ppo, 'w') as f:
    yaml.dump(cuopt_cfg_ppo, f)

In [ ]:
%%time
cuopt_main(config_path=tmp_cuopt_ppo)

### cuOpt with Tree-Based initialization

In [ ]:
with open('config_vrptw_cuopt_1000.yaml') as f:
    cuopt_cfg_tree = yaml.safe_load(f)

cuopt_cfg_tree['paths']['solutions']   = 'outputs/test_logs_vrptw_1000_tree.pkl'
cuopt_cfg_tree['names']['methods']      = ['Tree+cuOpt']
cuopt_cfg_tree['names']['main_method']  = 'Tree+cuOpt'
cuopt_cfg_tree['paths']['out_summary']  = 'test_summary_vrptw_1000_tree'

tmp_cuopt_tree = 'config_vrptw_cuopt_1000_tree_tmp.yaml'
with open(tmp_cuopt_tree, 'w') as f:
    yaml.dump(cuopt_cfg_tree, f)

In [ ]:
%%time
cuopt_main(config_path=tmp_cuopt_tree)

### Load and Display cuOpt Results

In [ ]:
fpath_ppo  = 'outputs/test_summary_vrptw_1000_ppo.pkl'
fpath_tree = 'outputs/test_summary_vrptw_1000_tree.pkl'

with open(fpath_ppo, 'rb') as f:
    cuopt_ppo_logs  = pkl.load(f)
with open(fpath_tree, 'rb') as f:
    cuopt_tree_logs = pkl.load(f)

df_cuopt_ppo  = cuopt_ppo_logs['summary'][df.columns]
df_cuopt_tree = cuopt_tree_logs['summary'][df.columns]

df_all = pd.concat([df, df_cuopt_ppo, df_cuopt_tree], ignore_index=True, sort=False)
df_all

### Full Comparison: PPO / Tree-Based / PPO+cuOpt / Tree+cuOpt

In [ ]:
print('=== Solution summary (VRPTW n=1000) ===')
summary = df_all.groupby('method').agg(
    mean_vehicles=('vehicles', 'mean'),
    mean_cost=('cost', 'mean')
).reset_index()
print(summary.to_string(index=False))

order = ['PPO', 'Tree-Based', 'PPO+cuOpt', 'Tree+cuOpt']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(data=df_all, x='method', y='vehicles', order=order, ax=axes[0])
axes[0].set_title('Mean # Vehicles - VRPTW n=1000')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=df_all, x='method', y='cost', order=order, ax=axes[1])
axes[1].set_title('Mean Cost - VRPTW n=1000')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('outputs/vrptw_1000_full_comparison.png', dpi=120)
plt.show()

best_cuopt_cost = df_all[df_all.method.isin(['PPO+cuOpt', 'Tree+cuOpt'])].groupby('problem_id').cost.min().mean()
ppo_cost_ratio  = df_ppo.cost.mean()  / best_cuopt_cost
tree_cost_ratio = df_tree.cost.mean() / best_cuopt_cost
print(f'\nPPO vs best cuOpt cost gap:        {100*(ppo_cost_ratio-1):.2f}%')
print(f'Tree-Based vs best cuOpt cost gap: {100*(tree_cost_ratio-1):.2f}%')

### Optimal solution summary (cuOpt injection)

The table below shows the best solution found by cuOpt per problem instance after RL initialization.

In [ ]:
best_cuopt_rows = (
    df_all[df_all.method.isin(['PPO+cuOpt', 'Tree+cuOpt'])]
    .sort_values('cost')
    .groupby('problem_id', as_index=False)
    .first()
    .rename(columns={'method': 'best_init'})
)

print('=== cuOpt optimal solution stats (VRPTW n=1000) ===')
print(f'  Mean vehicles: {best_cuopt_rows.vehicles.mean():.2f}')
print(f'  Mean cost:     {best_cuopt_rows.cost.mean():.4f}')
print(f'  Min cost:      {best_cuopt_rows.cost.min():.4f}')
print(f'  Max cost:      {best_cuopt_rows.cost.max():.4f}')
best_cuopt_rows[['problem_id', 'best_init', 'vehicles', 'cost']].head(20)

### Cleanup temporary config files

In [ ]:
for f in [tmp_cfg_ppo, tmp_cfg_tree, tmp_cuopt_ppo, tmp_cuopt_tree]:
    if os.path.exists(f):
        os.remove(f)
        print(f'Removed {f}')